# Exploratory Data Analysis - Bookings Dataset

In [23]:
import pandas as pd
import numpy as np

# Configuración de display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)


##  Load Data

In [24]:
df = pd.read_csv("/Users/helenagomez/Documents/booking-pipeline/data/raw/bookings.csv")
df.head(5)

,hotel,is_canceled,booking_to_arrival_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,board,country,market_segment,acquisition_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,parking_lot,total_of_special_requests,reservation_status,reservation_status_date
0,Apartment,0,277,2015,October,41,6,2,5,2,0.00,0,HB,TON,Associations,Contact,0,0,0,E,E,0,No Deposit,273.00,NaN,0,Transient-Party,77.29,0,0,Check-Out,2015-10-13
1,Hotel,1,109,2016,February,9,21,2,1,2,0.00,0,BB,SOM,Telephone Agency/Operator,Partners,0,1,0,A,A,0,Non Refund,19.00,NaN,44,Transient,75.00,0,0,Canceled,2015-12-18
2,Hotel,0,102,2015,October,42,16,0,2,2,0.00,0,HB,DEU,Telephone Agency/Operator,Partners,0,0,0,A,E,0,No Deposit,6.00,NaN,0,Transient-Party,109.00,0,0,Check-Out,2015-10-18
3,Apartment,1,35,2017,April,15,13,1,3,2,0.00,0,HB,SOM,Internet Agency,Partners,0,0,0,E,E,0,No Deposit,240.00,NaN,0,Transient,162.00,0,0,Canceled,2017-03-09
4,Hotel,1,164,2017,June,24,14,0,3,2,0.00,0,BB,GMB,Contact,Hotel,0,0,0,A,A,0,No Deposit,14.00,NaN,0,Transient,117.00,0,0,Canceled,2017-06-12


In [25]:
df.shape

(119390, 32)

## Data Quality

In [26]:
# Reemplazar 'NULL' por valores nulos reales si existen
df.replace("NULL", pd.NA, inplace=True)

# Nulos por columna
nulls = df.isna().sum().sort_values(ascending=False)
nulls[nulls > 0]

company     112593
agent        16340
country        488
children         4
dtype: int64

La columna company tiene un porcentaje muy alto de nulos, lo que sugiere que la mayoría de reservas no están asociadas a empresas. Evaluaría convertirla en variable indicadora o descartarla

## Tipos de hotel

In [27]:
df["hotel"].value_counts()

hotel
Hotel        79330
Apartment    40060
Name: count, dtype: int64

## Cancelaciones

In [28]:
cancel_rate = df["is_canceled"].mean()

print(f"Canceladas: {df['is_canceled'].sum():,}")
print(f"Tasa de cancelación: {cancel_rate:.2%}")

Canceladas: 44,224
Tasa de cancelación: 37.04%


La tasa de cancelación del 37% es significativa y sugiere una oportunidad clara para desarrollar modelos predictivos que permitan anticipar cancelaciones y optimizar estrategias de overbooking y pricing.

## Countries

In [29]:
countries = df["country"].value_counts()

print(f"Países únicos: {countries.shape[0]}")
countries.head(5)

Países únicos: 177


country
SOM    48590
TON    12129
GMB    10415
WLF     8568
DEU     7287
Name: count, dtype: int64

La distribución de países parece inconsistente con patrones turísticos reales, lo que sugiere que los códigos pueden estar anonimizados o que el dataset ha sido transformado.

## Análisis ADR (precio por noche)

In [30]:
df["adr"].describe()

count   119390.00
mean       101.83
std         50.54
min         -6.38
25%         69.29
50%         94.58
75%        126.00
max       5400.00
Name: adr, dtype: float64

El ADR tiene una media de 101€, pero la mediana es menor, lo que indica una distribución sesgada a la derecha. Existen outliers claros, como valores negativos y máximos extremadamente altos (5400€), que deberían analizarse y posiblemente tratarse antes de modelado.

* ADR promedio: €101.83 (un hotel de gama media)
* ADR máximo: €5,400 (una suite de lujo o una estancia especial)
* ADR mínimo: -€6.38 (sí, negativo — probablemente un ajuste o reembolso parcial)
* ADR = 0 en 1,959 reservas (huéspedes que no pagaron: cortesías, empleados, errores)

In [31]:
negatives = (df["adr"] < 0).sum()
zeros = (df["adr"] == 0).sum()

print(f"ADR negativos: {negatives}")
print(f"ADR en cero: {zeros}")

ADR negativos: 1
ADR en cero: 1959


Solo existe un ADR negativo, lo que indica un error puntual. Sin embargo, alrededor del 1.6% de reservas tienen ADR igual a cero. Esto podría deberse a promociones o ajustes contables, y requeriría validación antes de utilizar la variable en modelado.

## Feature Engineering - Total Nights

In [32]:
df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

zero_nights = (df["total_nights"] == 0).sum()
print(f"Reservas con 0 noches: {zero_nights}")

Reservas con 0 noches: 715


El 0.6% de reservas con 0 noches indica una inconsistencia lógica en los datos. Es probable que correspondan a cancelaciones tempranas o errores de registro. Antes de modelar, evaluaría si deben excluirse o tratarse específicamente.

## Distribution by year

In [33]:
df["arrival_date_year"].value_counts()

arrival_date_year
2016    56707
2017    40687
2015    21996
Name: count, dtype: int64

In [37]:
df.groupby("arrival_date_year")["is_canceled"].mean()

arrival_date_year
2015   0.37
2016   0.36
2017   0.39
Name: is_canceled, dtype: float64

La tasa de cancelación es bastante estable entre años, con una ligera tendencia al alza en 2017. Esto sugiere que la cancelación es un fenómeno estructural del negocio más que un evento puntual

## Ejemplos de Insights de Negocio

In [36]:
# Cancelación por tipo de hotel
pd.crosstab(df["hotel"], df["is_canceled"], normalize="index")

is_canceled,0,1
hotel,,
Apartment,0.72,0.28
Hotel,0.58,0.42


El tipo de alojamiento muestra una diferencia significativa en la tasa de cancelación: los hoteles presentan un 42% frente al 28% de los apartamentos. Esto sugiere que el tipo de alojamiento es una variable.

In [35]:
# ADR promedio por tipo de hotel
df.groupby("hotel")["adr"].mean()

hotel
Apartment    94.95
Hotel       105.30
Name: adr, dtype: float64

Los hoteles presentan un ADR promedio mayor y también una tasa de cancelación significativamente superior. Esto sugiere que el precio podría estar influyendo en la probabilidad de cancelación, o que el tipo de cliente asociado a hoteles es más volátil.

In [38]:
df.groupby("hotel").apply(lambda x: x.groupby("is_canceled")["adr"].mean())

/var/folders/cx/759xbkp92b54gf2qfs5pbc6r0000gn/T/ipykernel_55840/2987254691.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("hotel").apply(lambda x: x.groupby("is_canceled")["adr"].mean())


is_canceled,0,1
hotel,,
Apartment,90.79,105.79
Hotel,105.75,104.69


En apartamentos, los clientes que cancelan tienen un ADR significativamente mayor, lo que sugiere una posible relación entre precio y cancelación en este segmento. Sin embargo, en hoteles el ADR promedio es prácticamente igual entre canceladas y no canceladas, lo que indica que otros factores podrían estar influyendo más en ese caso.